[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnmarktaylor91/torchlens/blob/main/notebooks/learn/11_dynamic_graphs.ipynb)

In [1]:
# Colab setup. Does nothing when this notebook runs from a TorchLens checkout
# (learn_models.py is beside it). On Colab there is no checkout, so this installs
# TorchLens from the main branch and fetches learn_models.py next to the notebook.
import pathlib
import shutil
import subprocess
import sys
import urllib.request

if not pathlib.Path("learn_models.py").exists():
    REPO = "johnmarktaylor91/torchlens"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"torchlens[viz] @ git+https://github.com/{REPO}@main",
            "transformers",
            "ipywidgets",
        ],
        check=True,
    )
    if shutil.which("dot") is None:  # drawings need the Graphviz binary
        subprocess.run(["apt-get", "install", "-qq", "-y", "graphviz"], check=True)
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{REPO}/main/notebooks/learn/learn_models.py",
        "learn_models.py",
    )

# When the graph changes between forwards

**Level 4 of 5. Starting point: the graph changes between forwards.**

A trace records the path one input took through the model. Many models take a different path for
each input: a mixture of experts sends each token to a few of its experts, an early-exit network
stops at the first confident head, and any `if` on the input picks a branch. This notebook shows
what that does to a rerun of a trace, and the tool built for it: `tl.record`, which decides what to
keep op by op as the model runs, with no graph to build, so it follows whatever path each input
takes.

> **You'll want to know:** [`save=` and selectors](04_keep_what_you_need.ipynb) (`tl.func("relu")`,
> `tl.module("blocks.0")`), [recording without a graph](09_scale.ipynb) (`tl.record`). The
> [fast recorder tutorial](../fastlog_tutorial.ipynb) covers `tl.record` in more depth.

In [2]:
import warnings

import torch

import torchlens as tl

In [3]:
from learn_models import (
    early_exit,
    tiny_cnn,
    tiny_moe,
)  # tiny seeded CPU models; nothing downloads

moe, tokens = tiny_moe()  # three dense layers, then layers.3: a router and eight experts
_, other_tokens = tiny_moe(seed=1)  # a second batch of two token ids
print(tokens.tolist(), other_tokens.tolist())

[[12, 15]] [[5, 11]]


## 1. A trace reruns fast only on its own path

`trace.run(model, x)` captures a new input by reusing the graph the first trace recorded, which
costs a few plain forwards instead of a full trace. It checks every step against that graph. When
the new input sends its tokens to other experts, the steps no longer line up, so the rerun falls
back to a full capture: the answer is still right, it just costs a trace. `last_run` says which
engine ran and why the fast one refused, and a `ControlFlowDivergenceWarning` says the path
changed.

In [4]:
trace = tl.trace(moe, tokens, save=tl.module("unembed"))  # keep only the model's logits

same = trace.run(moe, tokens)  # same input, same experts
print("same input:   ", same.last_run["engine"], same.last_run["fast_refused"])

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    changed = trace.run(moe, other_tokens)  # other tokens reach other experts
print("other tokens: ", changed.last_run["engine"], changed.last_run["fast_refused"])
print("warned:       ", sorted({type(w.message).__name__ for w in caught}))

same input:    guarded_fast None
other tokens:  rerun conditional_arm_mismatch:fast_live_call_fingerprint
warned:        ['ControlFlowDivergenceWarning']


## 2. Record decides op by op

`tl.record` runs the model once and calls your rule on every operation as it happens. There is no
graph to match, so a different route is just a different set of operations. This rule keeps the
SiLU inside each expert, so the records name the experts that fired for this input. The model's
own output is untouched.

In [5]:
recording = tl.record(
    moe,
    tokens,
    save=lambda op: op.func_name == "silu" and "experts" in op.address,
)
fired = {r.ctx.address for r in recording}  # which experts ran for this input
print(sorted(fired))
print("output unchanged:", torch.equal(recording.output, moe(tokens)))

['layers.3.experts.2', 'layers.3.experts.7']
output unchanged: True


In [6]:
other = tl.record(
    moe,
    other_tokens,
    save=lambda op: op.func_name == "silu" and "experts" in op.address,
)
print(sorted({r.ctx.address for r in other}))  # another input, another set of experts
print(len(other), "records:", [r.ctx.label for r in other])

['layers.3.experts.4', 'layers.3.experts.5']
2 records: ['silu_1_27_raw', 'silu_2_35_raw']


Each record holds the saved tensor (`ram_payload`) beside the context the rule saw (`ctx`).

In [7]:
first = recording[0]
print(first.ctx.label, first.ctx.address, tuple(first.ram_payload.shape))

silu_1_27_raw layers.3.experts.2 (1, 32)


## 3. What a rule can see

The rule receives one context per operation. Beside `func_name` and `address` it carries the
module class, the stack of modules the operation runs inside, the output shape, and `recent_ops`,
the operations just before it, oldest first, so `recent_ops[-1]` is the previous operation.

In [8]:
contexts = []


def look(op):
    if op.func_name == "silu" and not contexts:
        contexts.append(op)  # keep the first SiLU's context to read below
    return False  # save nothing


tl.record(moe, tokens, save=look)
op = contexts[0]
print("module type:  ", op.module_type)
print("module stack: ", [frame.address for frame in op.module_stack])
print("shape:        ", op.shape)
print("recent ops:   ", [recent.func_name for recent in op.recent_ops])

module type:   Expert
module stack:  ['', 'layers.3', 'layers.3.experts.2']
shape:         (1, 32)
recent ops:    ['__eq__', 'nonzero', 'squeeze', '__getitem__', 'linear']


That is enough for "every ReLU after a convolution". In the tiny CNN the stem's convolution feeds
a ReLU directly, while each of the six blocks puts a BatchNorm between its convolution and its
ReLU, so of the seven ReLUs only the first matches.

In [9]:
cnn, image = tiny_cnn()  # stem conv and ReLU, six conv, BatchNorm, ReLU blocks
after_conv = tl.record(
    cnn,
    image,
    save=lambda op: op.func_name == "relu" and op.recent_ops[-1].func_name == "conv2d",
)
print(len(tl.record(cnn, image, save=tl.func("relu"))), "ReLUs;", len(after_conv), "after a conv")

7 ReLUs; 1 after a conv


`tl.preceded_by` says the same with selectors. It checks the operation's actual inputs among the
recent operations, not just whichever ran last, so it stays right when unrelated operations run in
between.

In [10]:
selected = tl.record(cnn, image, save=tl.func("relu") & tl.preceded_by(tl.func("conv2d")))
print([r.ctx.label for r in selected], [r.ctx.label for r in after_conv])

['relu_1_3_raw'] ['relu_1_3_raw']


## 4. Early exit, and stopping early

This model has three blocks, each followed by an exit head, and it returns at the first head that
is confident enough. An easy input leaves at the first exit; a hard one runs every block. A rule
that keeps the exit heads shows which ones ran.

In [11]:
model, x = early_exit()
inputs = {"easy": x * 8, "hard": x}  # the scaled input makes the first head confident


def exit_heads(op):
    return op.func_name == "linear" and op.address.startswith("exits.")


for name, xi in inputs.items():
    ran = [r.ctx.address for r in tl.record(model, xi, save=exit_heads)]
    print(f"{name}: {ran}")

easy: ['exits.0']
hard: ['exits.0', 'exits.1', 'exits.2']


`stop_after=` ends the forward pass once you have what you want. It is a rule too, called after
each event; here it fires when block 1 returns. The hard input stops there, and `is_stopped` says
so. The easy input left the model before block 1 ever ran, so nothing stopped it.

In [12]:
for name, xi in inputs.items():
    rec = tl.record(
        model,
        xi,
        save=exit_heads,
        stop_after=lambda op: op.kind == "module_exit" and op.address == "blocks.1",
    )
    print(f"{name}: stopped={rec.is_stopped}, kept {[r.ctx.address for r in rec]}")

easy: stopped=False, kept ['exits.0']
hard: stopped=True, kept ['exits.0']


## 5. Rules compose

Selectors and plain functions mix: `tl.where` turns any function of the context into a selector,
and `&` combines it with the others. This is section 2's rule as selectors.

In [13]:
in_an_expert = tl.where(lambda op: "experts" in (op.address or ""))
rec = tl.record(moe, tokens, save=tl.func("silu") & in_an_expert)
print(sorted({r.ctx.address for r in rec}) == sorted(fired))

True


## Where next

A trace is the right tool when the graph is the question: what feeds what, replaying each step,
editing one activation and following it downstream. `tl.record` is the right tool when the path
changes between inputs and you know what to keep. The [fast recorder
tutorial](../fastlog_tutorial.ipynb) covers saving to disk, interventions and backward passes with
`tl.record`, and [the fast paths guide](../../docs/guides/fast_paths.md) lists what each way of
capturing costs.